In [ ]:
## ----setup, include=FALSE---------------------------------------------------------------------------------------------------------------------------
knitr::opts_chunk$set(echo = TRUE)

In [ ]:
## ---------------------------------------------------------------------------------------------------------------------------------------------------
library(readr)
library(dplyr)
library(tidyr)
library(readxl)
library(writexl)
library(ggplot2)

In [ ]:
## ---------------------------------------------------------------------------------------------------------------------------------------------------
errors <- read_csv("F:/model_error_samples.csv")

In [ ]:
## ---------------------------------------------------------------------------------------------------------------------------------------------------
table_both <- errors %>%
  count(prompt_type, subject) %>%
  group_by(subject) %>%
  mutate(col_pct = n / sum(n)) %>%
  ungroup() %>%
  
  group_by(prompt_type) %>%
  mutate(total_n = sum(n)) %>%
  ungroup() %>%
  
  mutate(
    subject_label = sprintf("%d (%.1f%%)", n, 100*col_pct)
  ) %>%
  
  select(prompt_type, subject, subject_label, total_n) %>%
  
  pivot_wider(
    names_from = subject,
    values_from = subject_label,
    values_fill = "0 (0%)"
  ) %>%
  
  # add total % column
  mutate(
    Total = sprintf(
      "%d (%.1f%%)",
      total_n,
      100 * total_n / sum(total_n)
    )
  ) %>%
  select(prompt_type, Total, everything(), -total_n)

In [ ]:
table_both

In [ ]:
## ---------------------------------------------------------------------------------------------------------------------------------------------------
library(scales)

In [ ]:
plot_data <- errors %>%
  count(subject, prompt_type) %>%
  group_by(subject) %>%
  mutate(pct = n / sum(n))

In [ ]:
ggplot(plot_data,
       aes(x = subject,
           y = pct,
           fill = prompt_type)) +
  geom_col(position = "fill") +
  

  geom_text(
    aes(label = percent(pct, accuracy = 0.1)),
    position = position_fill(vjust = 0.5),
    size = 3
  ) +
  
  scale_y_continuous(labels = percent_format()) +
  labs(
    x = "Subject",
    y = "Percent of errors",
    fill = "Prompt type",
    title = "Distribution of error types within each subject"
  ) +
  theme_minimal()

# ---------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
analyse_errors <- function(input_path, output_path,
                           sheet_name = "selected_errors",
                           prompt_type_order = c("standard", "cot", "cot_few_shot", "rag", "rag_optional")) {

  error_analysis <- read_excel(input_path, sheet = sheet_name)

  # Replace NAs in a type-safe way:
  # - numeric/integer -> 0
  # - character       -> ""  (or "Unknown" if you prefer)
  # - logical         -> FALSE (optional)
  error_analysis <- error_analysis %>%
    mutate(across(where(is.numeric), ~ replace_na(., 0))) %>%
    mutate(across(where(is.character), ~ replace_na(., ""))) %>%
    mutate(across(where(is.logical), ~ replace_na(., FALSE)))

  error_counts <- error_analysis %>%
    count(prompt_type, shot_n, k, `Error Category`, name = "count") %>%
    group_by(prompt_type, shot_n, k) %>%
    mutate(percentage = round(100 * count / sum(count), 2)) %>%
    ungroup()

  error_pivot <- error_counts %>%
    select(prompt_type, shot_n, k, `Error Category`, percentage) %>%
    pivot_wider(
      names_from = `Error Category`,
      values_from = percentage,
      values_fill = 0
    ) %>%
    mutate(prompt_type = factor(prompt_type, levels = prompt_type_order)) %>%
    arrange(prompt_type, shot_n, k)

  # If error category columns are numeric-coded, sort them numerically
  id_cols <- c("prompt_type", "shot_n", "k")
  cat_cols <- setdiff(names(error_pivot), id_cols)
  cat_nums <- suppressWarnings(as.numeric(cat_cols))
  if (all(!is.na(cat_nums))) {
    error_pivot <- error_pivot %>%
      select(all_of(id_cols), all_of(cat_cols[order(cat_nums)]))
  }

  writexl::write_xlsx(error_pivot, path = output_path)
  error_pivot
}

In [ ]:
# Run
input_path  <- "F:/model_error_analysis.xlsx"
output_path <- "F:/error_percentages.xlsx"

In [ ]:
error_pivot <- analyse_errors(input_path, output_path)

# ---------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
data <- read_excel("F:/error_percentages.xlsx")

In [ ]:
data <- data %>%
  mutate(config = paste(prompt_type,
                        paste0("shot=", shot_n),
                        paste0("k=", k),
                        sep = " | "))

In [ ]:
long_data <- data %>%
  pivot_longer(
    cols = `0`:`8`,
    names_to = "Error_Category",
    values_to = "percentage"
  )
error_labels <- c(
  "0" = "Database error",
  "1" = "Wrong fact",
  "2" = "Partially correct",
  "3" = "Missing context",
  "4" = "Question misunderstood",
  "5" = "Logic error",
  "6" = "LLM generation error",
  "7" = "Reasoning incomplete",
  "8" = "Answer–reasoning mismatch"
)

In [ ]:
long_data <- long_data %>%
  mutate(
    Error_Category = factor(
      Error_Category,
      levels = names(error_labels),
      labels = error_labels
    )
  )

HEATMAP

In [ ]:
ggplot(long_data,
       aes(x = Error_Category,
           y = config,
           fill = percentage)) +
  geom_tile(color = "white") +
  
  # add % labels inside tiles
  geom_text(aes(label = sprintf("%.1f", percentage)),
            size = 3) +
  
  scale_fill_gradient(low = "white", high = "red") +
  labs(
    title = "Error-category distribution by prompting configuration",
    x = "Error category",
    y = "Prompt configuration",
    fill = "%"
  ) +
  theme_minimal() +
  theme(
    axis.text.y = element_text(size = 8),
    axis.text.x = element_text(angle = 45, hjust = 1)
  )

FIGURE B: STACKED BAR (aggregated by prompt_type)

In [ ]:
agg <- long_data %>%
  group_by(prompt_type, Error_Category) %>%
  summarise(percentage = mean(percentage), .groups = "drop")

In [ ]:
ggplot(agg,
       aes(x = prompt_type,
           y = percentage,
           fill = Error_Category)) +
  geom_col() +
  
  # add percentage labels
 geom_text(
  aes(label = ifelse(percentage >= 5,
                     sprintf("%.1f%%", percentage), "")),
  position = position_stack(vjust = 0.5),
  size = 3
)
 
  
  labs(
    title = "Error composition by prompting strategy",
    x = "Prompt type",
    y = "Percentage",
    fill = "Error category"
  ) +
  theme_minimal()

# ---------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
raw <- read_excel("F:/model_error_analysis.xlsx",
                  sheet = "selected_errors")

In [ ]:
raw <- raw %>%
  mutate(across(where(is.numeric), ~ replace_na(., 0)))

In [ ]:
subject_data <- raw %>%
  count(subject, `Error Category`) %>%
  group_by(subject) %>%
  mutate(pct = 100 * n / sum(n)) %>%
  ungroup()

In [ ]:
# error labels
error_labels <- c(
  "0" = "Database error",
  "1" = "Wrong fact",
  "2" = "Partially correct",
  "3" = "Missing context",
  "4" = "Question misunderstood",
  "5" = "Logic error",
  "6" = "LLM generation error",
  "7" = "Reasoning incomplete",
  "8" = "Answer–reasoning mismatch"
)

In [ ]:
subject_data <- subject_data %>%
  mutate(
    `Error Category` = factor(
      `Error Category`,
      levels = names(error_labels),
      labels = error_labels
    )
  )

In [ ]:
ggplot(subject_data,
       aes(x = subject,
           y = pct,
           fill = `Error Category`)) +
  geom_col() +
  

  geom_text(
    aes(label = sprintf("%.1f%%", pct)),
    position = position_stack(vjust = 0.5),
    size = 3
  ) +
  
  labs(
    title = "Error composition by subject",
    x = "Subject",
    y = "Percentage",
    fill = "Error category"
  ) +
  theme_minimal()